# CRE3 - Assignment 2: Parametric study of $CO_2$/$CO$ hydrogenation and DME synthesis

Group 6: 

Julian Stierstorfer (1160552)\
Ivana Garzon Casanova ()\
Venkata Uda ()

## 1. Introduction

## 2. Reaction network and key reactions

The reaction system that describes the hydrogenation of $CO_2$ and $CO$ as well as the synthesis of DME is given as follows:

\begin{align}
CO_2 + 3H_2 &\overset{R_1}{\rightleftharpoons} CH_3OH + H_2O\\
CO + 2H_2 &\overset{R_2}{\rightleftharpoons} CH_3OH\\
CO_2 + H_2 &\overset{R_3}{\rightleftharpoons} CO + H_2O\\
2CH_3OH &\overset{R_4}{\rightleftharpoons} CH_3OCH_3 + H_2O
\end{align}

Equations (1) and (2) describe the hydrogenation of $CO_2$ and $CO$ while forming methanol ($CH_3OH$). Reaction (3) shows the reverse water-gas shift reaction, while reaction (4) describes the formation of DME from methanol.

### 2.1 Matrix of stoichiometric coefficients

From the reaction equations the Matrix of stoichiometric components can be derived by sorting all components and reactions:

\begin{equation}
  \underline{N} =
    \begin{bmatrix}
          & R_1 & R_2 & R_3 & R_4\\
      CO_2      & -1 & 0  & -1 & 0\\
      H_2       & -3 & -2 & -1 & 0\\
      CH_3OH    & 1  & 1  & 0  & -2\\
      H_2O      & 1  & 0  & 1  & 1\\
      CO        & 0  & -1 & 1  & 0\\
      CH_3OCH_3 & 0  & 0  & 0  & 1
    \end{bmatrix}
\end{equation}

### 2.2 Determining key reactions and components
The number of key reactions and components $R_{\nu}$ is calculated in order to obtain a minimum independent description of the reaction network. Once key reactions and components are known, the linear independent part of the reaction network can be used for a simpler calculation. In practice this also means that by measuring the concentration of key components at a given time, concentration of all other components can be calculated. 

The number of key reactions and components can be calculated as follows [1]:
\begin{equation}
R_\nu = \operatorname{rank}(\underline{N})
\end{equation} 

The rank of $\underline{N}$ can in principle be calculated by applying the Gauss algorithm by hand however this can be done numerically using the `matrix_rank` function by the `linalg`-package by `numpy`. Therefore the initial matrix $\underline{N}_{init}$ is implemented and its rank is calculated.

In [6]:
# importing needed packages
import numpy as np # general numpy package
from numpy.linalg import matrix_rank # linear algebra package for matrix rank
import matplotlib.pyplot as plt # plotting package


# defining labels for rows (components) and  columns (reactions)
components = ["CO2", "H2", "CH3OH", "H2O", "CO", "CH3OCH3"]
reactions = ["R1", "R2", "R3", "R4"]

N_init = np.array([
    [-1,  0, -1,  0],   # CO2
    [-3, -2, -1,  0],   # H2
    [ 1,  1,  0, -2],   # CH3OH
    [ 1,  0,  1,  1],   # H2O
    [ 0, -1,  1,  0],   # CO
    [ 0,  0,  0,  1]    # CH3OCH3
], dtype=float) # initial matrix of stoichiometric coefficients for the reactions

# Calculate the rank of the stoichiometric matrix
rank_N = matrix_rank(N_init)
print(f"Rank of the stoichiometric matrix N: {rank_N}")



Rank of the stoichiometric matrix N: 3


As obtained from the numerical calculation the rank of the matrix is found. $R_{\nu} = 3$. Therefore, three key components are required. Their selection should consider both practical measurability and chemical relevance, meaning that the chosen components should be experimentally accessible and meaningful for describing the reaction network.

In this case the key components are choosen to be $CO_2$, $CH_3OH$ and $DME$. On the one hand $CO_2$ is one of the carbon feeding species that takes place in hydrogenation, on the other hand $CH_3OH$ and $DME$ represent a critical intermediate as well as the product component.

As well as taking into account key components, key reactions can be determined as well. According to the rank of $\underline{N}$, 3 linear independent key reactions are sufficient to describe the reaction system. After further investigation it can be found that $R_3 = R_1 - R_2$. Therefore the key reactions of the system are $R_1, R_2$ and $R_4$.

Thus the matrix of stoichiometric components can be rearranged according to the key components and reactions:

In [7]:
# Rearranged matrix
# key components first: CO2, CH3OH, CH3OCH3
# remaining components: H2, H2O, CO
row_order = [0, 2, 5, 1, 3, 4]

# independent key reactions first: R1, R2, R4
# dependent reaction R3 last
col_order = [0, 1, 3, 2]

N = N_init[row_order, :][:, col_order]

component_order = ["CO2", "CH3OH", "CH3OCH3", "H2", "H2O", "CO"]
reaction_order = ["R1", "R2", "R4", "R3"]

print("Rearranged matrix N:")
print(N)

print("\nComponent order:")
print(component_order)

print("\nReaction order:")
print(reaction_order)

print("\nRank of rearranged matrix:")
print(matrix_rank(N))

Rearranged matrix N:
[[-1.  0.  0. -1.]
 [ 1.  1. -2.  0.]
 [ 0.  0.  1.  0.]
 [-3. -2.  0. -1.]
 [ 1.  0.  1.  1.]
 [ 0. -1.  0.  1.]]

Component order:
['CO2', 'CH3OH', 'CH3OCH3', 'H2', 'H2O', 'CO']

Reaction order:
['R1', 'R2', 'R4', 'R3']

Rank of rearranged matrix:
3


The rearranged matrix is now given as:

\begin{equation}
  \underline{N} =
    \begin{bmatrix}
          & R_1 & R_2 & R_4 & R_3\\
      CO_2      & -1 & 0  & 0  & -1\\
      CH_3OH    & 1  & 1  & -2 & 0\\
      CH_3OCH_3 & 0  & 0  & 1  & 0\\
      H_2       & -3 & -2 & 0  & -1\\
      H_2O      & 1  & 0  & 1  & 1\\
      CO        & 0  & -1 & 0  & 1
    \end{bmatrix}
\end{equation}

This matrix can now be partitioned into four submatrices:

\begin{equation}
\underline{N} =
\begin{bmatrix}
\underline{N}_{1,1} & \underline{N}_{1,2}\\
\underline{N}_{2,1} & \underline{N}_{2,2}
\end{bmatrix} =
\left[
\begin{array}{ccc|c}
-1 & 0  & 0  & -1\\
 1 & 1  & -2 & 0\\
 0 & 0  & 1  & 0\\
\hline
-3 & -2 & 0  & -1\\
 1 & 0  & 1  & 1\\
 0 & -1 & 0  & 1
\end{array}
\right]
\end{equation}





PLAUSIBILITY CHECK OF MASS BALANCE AT THE END

# Sources

[1] Güttel, R., & Turek, T. (2021). *Chemische Reaktionstechnik*. Springer Spektrum. https://doi.org/10.1007/978-3-662-63150-8